# Funko pop analyzer

In [20]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast

В този проект ще се опитаме първо да изчистим данните за фукно поп фигурки и данни за филми, след това ще опитаме да ги обединим като ще се опитаме да проверим няколко хипотези
Идеи за хипотези:
H1: Филми с по-висок рейтинг (IMDb/Rotten Tomatoes) имат повече Funko Pop продукти
H2: Box office приходите са по-силен фактор за броя фигурки от рейтинга
H3: Филми от franchise (Marvel, Star Wars и т.н.) имат непропорционално повече фигурки спрямо standalone филми
H4: Определени жанрове (action, fantasy, animation) имат значително повече Funko Pop фигурки
H5: Horror филмите имат по-малко фигурки, но по-висока „нишова концентрация“ (по-малко филми, но с много фигурки)
H6: По-новите филми имат повече Funko Pop фигурки (заради растежа на бранда)
H7: Има пик в производството на фигурки около годината на излизане на филма
H8: Филми от определени студиа (Disney, Warner Bros) имат повече фигурки
H9: Collaboration ефект – когато има crossover (напр. Marvel), броят на фигурките расте експоненциално

In [23]:
df = pd.read_csv('funko.csv')

In [5]:
df

,uid,title,product_type,price,interest,license,tags,vendor,form_factor,feature,related,description,gid,created_at,published_at,updated_at,handle,img
0,7051374362813,"""It's Crunch Time "" Kids Tee - General Mills",Apparel,7.0,['Ad Icons'],['General Mills'],"['Apparel', 'Cereal Day', 'Kids Tee', 'Markdow...",Funko Pop Up Shop,[],[],"['7051374788797', '7051374854333', '7051374723...","<p>""It's Crunch Time"" with the Count Chocula P...",gid://shopify/Product/7051374362813,2021-10-29T00:02:52-00:00,2021-10-29T16:30:00-00:00,2022-05-23T16:50:36-00:00,ad-icons-general-mills-its-crunch-time-kids-bl...,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1,7231381405885,"""This is the Way"" Kids Tee - The Mandalorian",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'Kids Tee', 'May the 4th', 'May th...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381405885,2022-05-03T20:57:34-00:00,2022-05-04T15:30:00-00:00,2022-05-27T13:25:49-00:00,star-wars-this-is-the-way-kids-purple-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
2,7231381536957,"""This is the Way"" Neon Blast Kids Tee - The Ma...",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'May the 4th', 'May the 4th Be Wit...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381536957,2022-05-03T20:57:40-00:00,2022-05-04T15:30:01-00:00,2022-05-29T18:15:32-00:00,star-wars-the-mandalorian-this-is-the-way-kids...,https://cdn.shopify.com/s/files/1/1052/2158/pr...
3,7051374395581,"""Time for a Midnight Bite"" Tee - General Mills",Apparel,14.0,['Ad Icons'],['General Mills'],"['Apparel', 'Cereal Day', 'Markdown Item', 'Sa...",Funko Pop Up Shop,[],[],"['7051374788797', '7051374854333', '7051374723...","<p>Like the Pop! Tee says, it's ""Time for a Mi...",gid://shopify/Product/7051374395581,2021-10-29T00:02:57-00:00,2021-10-29T16:30:01-00:00,2022-05-26T04:04:45-00:00,ad-icons-general-mills-midnight-bite-black-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
4,7231381668029,"""Where He Goes, I Go"" Grogu Kids Tee - The Man...",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'May the 4th', 'May the 4th Be Wit...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381668029,2022-05-03T20:57:45-00:00,2022-05-04T15:30:02-00:00,2022-05-30T02:40:26-00:00,star-wars-where-he-goes-i-go-grogu-kids-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,6958000734397,Zombie Hunter Spidey - What If…?,Pop!,12.0,['Marvel'],"['Spider-Man', 'What If']","['Disney Plus', 'FFCBDAY', 'Gift Box', 'Gift G...",Funko Shop,[],['Back In Stock'],"['6218636001469', '7078124683453', '7193953960...","In the endless possibilities, venture to wonde...",gid://shopify/Product/6958000734397,2021-09-09T23:43:24-00:00,2022-05-26T16:41:07-00:00,2022-05-31T08:01:25-00:00,marvel-what-if-zombie-hunter-spidey-pop,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1478,6958000963773,Zombie Iron Man - What If…?,Pop!,12.0,['Marvel'],"['Iron Man', 'What If']","['Disney Plus', 'FFCBDAY', 'Gift Box', 'Gift G...",Funko Shop,[],[],"['6958000734397', '7078124683453', '6924450332...","In the endless possibilities, venture to wonde...",gid://shopify/Product/6958000963773,2021-09-09T23:43:37-00:00,2021-11-22T23:44:03-00:00,2022-05-31T00:50:26-00:00,marvel-what-if-zombie-iron-man-pop,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1479,7232416514237,Zorro,Pop!,12.0,['Classics'],['Zorro'],"['FFCBDAY', 'Gift Box', 'NOTIFY', 'PPP']",Funko Shop,[],['Coming Soon'],"['6161453613245', '7193953599677', '7239037386...",Celebrate the 65th Anniversary of <i>Zorro</i>...,gid://shopify/Product/7232416514237,2022-05-04T23:54:25-00:00,2022-05-05T16:00:01-00:00,2022-05-05T16:00:05

1. Трябва всички стойности от полето title да се уеднаквят - в момента има такива с кавички, без кавички, трябва да се направят всички на lowercase, 

In [24]:
# ---------- CONFIG ----------
MULTI_COLS = ["interest", "license", "tags"]
DATE_COLS  = ["created_at", "published_at", "updated_at"]
DROP_COLS  = ["gid", "handle", "img"]

# ---------- 0) SAFE text hygiene ----------
# ONLY for true text columns (NOT list columns)
TEXT_COLS = [c for c in df.select_dtypes(include="object").columns
             if c not in MULTI_COLS]

for c in TEXT_COLS:
    df[c] = df[c].astype(str).str.strip()


# ---------- 1) FIX list-like columns properly ----------
def clean_list_cell(x):
    if pd.isna(x):
        return ""

    # already list
    if isinstance(x, list):
        vals = x

    else:
        try:
            parsed = ast.literal_eval(x)
            vals = parsed if isinstance(parsed, list) else [parsed]
        except:
            # fallback for dirty strings
            x = str(x).strip().strip("[]")
            vals = [i.strip().strip("'").strip('"') for i in x.split(",") if i.strip()]

    # remove garbage values
    vals = [v for v in vals if v and str(v).lower() != "nan"]

    return ", ".join(vals)


for c in MULTI_COLS:
    if c in df.columns:
        df[c] = df[c].apply(clean_list_cell)


# ---------- 2) Split multi-value fields into columns ----------
def split_wide(df, source_col, prefix):
    if source_col not in df.columns:
        return df

    tmp = df[source_col].fillna("")

    max_items = tmp.apply(lambda s: 0 if s == "" else len(s.split(","))).max()
    if max_items <= 1:
        return df

    parts = tmp.str.split(",", expand=True)
    parts = parts.apply(lambda col: col.str.strip())

    parts.columns = [f"{prefix}_{i+1}" for i in range(parts.shape[1])]

    return pd.concat([df, parts], axis=1)


for col in MULTI_COLS:
    df = split_wide(df, col, col.capitalize())


# ---------- 3) Clean HTML ----------
if "description" in df.columns:
    df["description"] = (
        df["description"]
        .astype(str)
        .str.replace(r"<.*?>", "", regex=True)
        .str.strip()
    )


# ---------- 4) Datetime parsing ----------
for c in DATE_COLS:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)


# ---------- 5) Drop unused columns ----------
df_clean = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")


# ---------- 6) Long format (tags exploded) ----------
if "tags" in df.columns:
    long_tags = (
        df[["uid", "tags"]]
        .assign(tags=lambda d: d["tags"].str.split(","))
        .explode("tags")
    )

    long_tags["tags"] = long_tags["tags"].astype(str).str.strip()
    long_tags = long_tags[long_tags["tags"] != ""]
    long_tags.to_csv("data_tags_long.csv", index=False)


# ---------- 7) Save ----------
df_clean.to_csv("data_cleaned_wide.csv", index=False)

print("Saved: data_cleaned_wide.csv")

Saved: data_cleaned_wide.csv


In [15]:
df

,uid,title,product_type,price,interest,license,tags,vendor,form_factor,feature,related,description,gid,created_at,published_at,updated_at,handle,img
0,7051374362813,"""It's Crunch Time "" Kids Tee - General Mills",Apparel,7.0,['Ad Icons'],['General Mills'],"['Apparel', 'Cereal Day', 'Kids Tee', 'Markdow...",Funko Pop Up Shop,[],[],"['7051374788797', '7051374854333', '7051374723...","<p>""It's Crunch Time"" with the Count Chocula P...",gid://shopify/Product/7051374362813,2021-10-29T00:02:52-00:00,2021-10-29T16:30:00-00:00,2022-05-23T16:50:36-00:00,ad-icons-general-mills-its-crunch-time-kids-bl...,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1,7231381405885,"""This is the Way"" Kids Tee - The Mandalorian",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'Kids Tee', 'May the 4th', 'May th...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381405885,2022-05-03T20:57:34-00:00,2022-05-04T15:30:00-00:00,2022-05-27T13:25:49-00:00,star-wars-this-is-the-way-kids-purple-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
2,7231381536957,"""This is the Way"" Neon Blast Kids Tee - The Ma...",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'May the 4th', 'May the 4th Be Wit...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381536957,2022-05-03T20:57:40-00:00,2022-05-04T15:30:01-00:00,2022-05-29T18:15:32-00:00,star-wars-the-mandalorian-this-is-the-way-kids...,https://cdn.shopify.com/s/files/1/1052/2158/pr...
3,7051374395581,"""Time for a Midnight Bite"" Tee - General Mills",Apparel,14.0,['Ad Icons'],['General Mills'],"['Apparel', 'Cereal Day', 'Markdown Item', 'Sa...",Funko Pop Up Shop,[],[],"['7051374788797', '7051374854333', '7051374723...","<p>Like the Pop! Tee says, it's ""Time for a Mi...",gid://shopify/Product/7051374395581,2021-10-29T00:02:57-00:00,2021-10-29T16:30:01-00:00,2022-05-26T04:04:45-00:00,ad-icons-general-mills-midnight-bite-black-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
4,7231381668029,"""Where He Goes, I Go"" Grogu Kids Tee - The Man...",Apparel,10.0,['Star Wars'],['Star Wars'],"['Apparel', 'May the 4th', 'May the 4th Be Wit...",Funko Shop,[],[],"['7254814130365', '7254814064829', '4491928240...",<p>Celebrate May the Fourth in stellar style w...,gid://shopify/Product/7231381668029,2022-05-03T20:57:45-00:00,2022-05-04T15:30:02-00:00,2022-05-30T02:40:26-00:00,star-wars-where-he-goes-i-go-grogu-kids-tee,https://cdn.shopify.com/s/files/1/1052/2158/pr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1477,6958000734397,Zombie Hunter Spidey - What If…?,Pop!,12.0,['Marvel'],"['Spider-Man', 'What If']","['Disney Plus', 'FFCBDAY', 'Gift Box', 'Gift G...",Funko Shop,[],['Back In Stock'],"['6218636001469', '7078124683453', '7193953960...","In the endless possibilities, venture to wonde...",gid://shopify/Product/6958000734397,2021-09-09T23:43:24-00:00,2022-05-26T16:41:07-00:00,2022-05-31T08:01:25-00:00,marvel-what-if-zombie-hunter-spidey-pop,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1478,6958000963773,Zombie Iron Man - What If…?,Pop!,12.0,['Marvel'],"['Iron Man', 'What If']","['Disney Plus', 'FFCBDAY', 'Gift Box', 'Gift G...",Funko Shop,[],[],"['6958000734397', '7078124683453', '6924450332...","In the endless possibilities, venture to wonde...",gid://shopify/Product/6958000963773,2021-09-09T23:43:37-00:00,2021-11-22T23:44:03-00:00,2022-05-31T00:50:26-00:00,marvel-what-if-zombie-iron-man-pop,https://cdn.shopify.com/s/files/1/1052/2158/pr...
1479,7232416514237,Zorro,Pop!,12.0,['Classics'],['Zorro'],"['FFCBDAY', 'Gift Box', 'NOTIFY', 'PPP']",Funko Shop,[],['Coming Soon'],"['6161453613245', '7193953599677', '7239037386...",Celebrate the 65th Anniversary of <i>Zorro</i>...,gid://shopify/Product/7232416514237,2022-05-04T23:54:25-00:00,2022-05-05T16:00:01-00:00,2022-05-05T16:00:05